# Diffusion Models
In this lab scenario, you will implement a simplified diffusion model and train it on MNIST.  
The scenario is based on [Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239) paper.

## Intro

### What are the diffusion models?
Briefly speaking they are trained to remove different levels of Gaussian noise from the data.  
To be more precise  
Let $D$ be our diffusion model  
Let $q(x_t|x_{t-1}) = \mathcal{N}(\sqrt{1 - \beta_t}x_{t-1}, \beta_t \mathbb{1})$  

We will train our diffusion model as follows:  
1. We will take a random image $x_0$ from the dataset (for example image of a random flower)
2. Sample $t$ - number of "noise applications"
3. We will use $q$ to add $t$ layers of noise to $x_0$
4. We will ask $D(x_t, t)$ to predict the noise


### More details
How will we exactly add the noise to $x_0$?  
First of all, we note that we can write sampling of $x_1$ as follows:  
$\sqrt{1 - \beta_1}x_0 + \epsilon \sqrt{\beta_1}$  
where $\epsilon$ ~ $\mathcal{N}(0, \mathbb{1})$   
Then we note that we can write $x_2$ as follows:  
$(\sqrt{1 - \beta_1}x_0 + \epsilon \sqrt{\beta_1})\sqrt{1 - \beta_2} + \epsilon'\sqrt{\beta_2}$  
$\sqrt{(1-\beta_1) (1-\beta_2)}x_0 + \epsilon \sqrt{\beta_1(1-\beta_2)} + \epsilon'\sqrt{\beta_2}$   
as $\epsilon$ and $\epsilon'$ are independent we can sum variances and as we are dealing with Gaussian distributions here so we can write  
$\sqrt{(1-\beta_1) (1-\beta_2)}x_0 + \epsilon'' \sqrt{(\beta_1 + \beta_2 - \beta_1\beta_2)}$  

If we define that
 $\alpha_t = (1-\beta_1) (1-\beta_2) \cdots (1-\beta_t)$  
then we have that
$\sqrt{\alpha_2}x_0 + \epsilon'' \sqrt{1 - \alpha_2}$  

In general, we can get that  
$x_t$ ~ $\mathcal{N}(\sqrt{\alpha_t}x_0, (1-\alpha_t) \mathbb{1})$  


So we will create $x_t$ as follows:  
$x_t = \sqrt{\alpha_t}x_0 +  \epsilon \sqrt{1 - \alpha_t}$   
where $\epsilon$ ~ $\mathcal{N}(0, \mathbb{1})$   
and $D$ will be tasked with predicting $\epsilon$ given $x_t$ and $t$.

For more details you can refer to [Understanding Diffusion Models: A Unified Perspective](https://arxiv.org/abs/2208.11970).


## Preparation
We import packages and download the data.

### Imports

In [ ]:
import torchvision
import torch
from typing import List, Tuple
from tqdm import tqdm
from matplotlib import pyplot as plt

### Data
We will use a subset of MNIST data with simple augmentations.  
This is mainly to speed up the training, you can later try the whole dataset.

In [ ]:
DATA = "DEBUG"  # adjust after completing the task

if DATA == "DEBUG" or DATA == "MNIST":
    DATA_PATH = "~/torch_datasets/mnist"
    IMAGE_CHANNELS = 1

    input_transforms = torchvision.transforms.Compose(
        [
            torchvision.transforms.Resize((16, 16) if DATA == "DEBUG" else (32, 32)),
            torchvision.transforms.ToTensor(),  # our input is an image
            torchvision.transforms.RandomRotation(22.5),
        ]
    )
    TRAIN_DATASET = torchvision.datasets.MNIST(
        root=DATA_PATH, train=True, download=True, transform=input_transforms
    )

    if DATA == "DEBUG":
        subset = torch.logical_or(
            TRAIN_DATASET.targets == 1, TRAIN_DATASET.targets == 7
        )
        TRAIN_DATASET.data, TRAIN_DATASET.targets = (
            TRAIN_DATASET.data[subset],
            TRAIN_DATASET.targets[subset],
        )
elif DATA == "CIFAR_P":
    DATA_PATH = "~/torch_datasets/cifar10"
    IMAGE_CHANNELS = 3
    input_transforms = torchvision.transforms.Compose(
        [
            torchvision.transforms.Resize((32, 32)),
            torchvision.transforms.ToTensor(),  # our input is an image
        ]
    )
    TRAIN_DATASET = torchvision.datasets.CIFAR10(
        root=DATA_PATH, train=True, download=True, transform=input_transforms
    )
    tensor_targets = torch.tensor(TRAIN_DATASET.targets)
    subset = tensor_targets == 0
    TRAIN_DATASET.data, TRAIN_DATASET.targets = (
        TRAIN_DATASET.data[subset],
        tensor_targets[subset],
    )
else:
    raise ValueError(f"Data {DATA} not supported")


BATCH_SIZE = 64
TRAIN_LOADER = torch.utils.data.DataLoader(
    TRAIN_DATASET, shuffle=True, batch_size=BATCH_SIZE
)

## Architecture
We are going to use a simplified U-Net variant with time encoding.  
We will also make several simplifications in comparison to the architecture used in [Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239).  
In particular, we omit the attention.

### Residual Block with Time
Finish the implementation of `ResidualBlockWithTime`.  
The main flow can be described as follows:  
1. Process input using `GroupNorm` (with `in_channels` groups), `SiLU`, and `Conv2d` creating `x`
2. Process `t` using a `Linear` layer and reshape it to `(BATCH, out_channels, 1, 1)`.
3. Add `t` to `x` creating `xm`
4. Process `xm` using `GroupNorm`, `SiLU`, `Dropout` and `Conv2d`
5. Add channel adjusted input to `xm`  

Hint: the official source code in TensorFlow is available [here](https://github.com/hojonathanho/diffusion/blob/1e0dceb3b3495bbe19116a5e1b3596cd0706c543/diffusion_tf/models/unet.py#L37)

In [ ]:
class ResidualBlockWithTime(torch.nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        num_groups: int,
        time_features: int,
        dropout: float,
    ):
        super().__init__()

        self.block_1 = torch.nn.Sequential(
            torch.nn.GroupNorm(num_groups=in_channels, num_channels=in_channels),
            torch.nn.SiLU(),
            torch.nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                padding=1,
            ),
        )

        self.time_handling = torch.nn.Sequential(
            torch.nn.SiLU(),
            torch.nn.Linear(in_features=time_features, out_features=out_channels),
        )

        self.block_2 = torch.nn.Sequential(
            torch.nn.GroupNorm(num_groups=num_groups, num_channels=out_channels),
            torch.nn.SiLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Conv2d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=3,
                padding=1,
            ),
        )

        if in_channels == out_channels:
            self.adjust_residual = torch.nn.Identity()
        else:
            self.adjust_residual = torch.nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=1,
            )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        assert len(x.shape) == 4  # B C H W
        assert len(t.shape) == 2  # B F

        ### TODO {

        ### }

        assert result.shape == x.shape

        return result


@torch.no_grad()
def test_residual_block():
    x = torch.randn(2, 4, 5, 7)
    t = torch.randn(2, 7)
    res1 = ResidualBlockWithTime(
        in_channels=4, out_channels=12, time_features=7, num_groups=2, dropout=0.1
    )
    res2 = ResidualBlockWithTime(
        in_channels=12, out_channels=12, time_features=7, num_groups=3, dropout=0.0
    )

    assert res2(res1(x, t), t).shape == (2, 12, 5, 7)


test_residual_block()

### Downsample and Upsample blocks
Downsample and Upsample blocks are already implemented

In [ ]:
class DownsampleBlock(torch.nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        num_groups: int,
        time_features: int,
        dropout: float,
        num_residual_blocks: int = 1,
    ):
        super().__init__()
        assert num_residual_blocks > 0

        residual_blocks = [
            ResidualBlockWithTime(
                in_channels=in_channels,
                out_channels=out_channels,
                num_groups=num_groups,
                time_features=time_features,
                dropout=dropout,
            )
        ]

        residual_blocks = residual_blocks + [
            ResidualBlockWithTime(
                in_channels=out_channels,
                out_channels=out_channels,
                num_groups=num_groups,
                time_features=time_features,
                dropout=dropout,
            )
            for _ in range(num_residual_blocks - 1)
        ]

        self.residual_blocks = torch.nn.ModuleList(residual_blocks)

        self.downsample = torch.nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(
        self, x: torch.Tensor, t: torch.Tensor
    ) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        assert len(x.shape) == 4  # B C H W
        assert len(t.shape) == 2  # B F

        skips = []
        for rb in self.residual_blocks:
            x = rb(x, t)
            skips.append(x)

        x = self.downsample(x)

        return x, skips


@torch.no_grad()
def test_downsample_block():
    x = torch.randn(2, 4, 8, 18)
    t = torch.randn(2, 7)
    db = DownsampleBlock(
        in_channels=4,
        out_channels=12,
        time_features=7,
        num_groups=2,
        dropout=0.1,
        num_residual_blocks=5,
    )

    y, skips = db(x, t)

    assert y.shape == (2, 12, 4, 9)
    assert len(skips) == 5


test_downsample_block()

In [ ]:
class UpsampleBlock(torch.nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        num_groups: int,
        time_features: int,
        dropout: float,
        num_residual_blocks: int,
    ):
        super().__init__()
        assert num_residual_blocks > 0
        residual_blocks = [
            ResidualBlockWithTime(
                in_channels=in_channels + in_channels,
                out_channels=out_channels,
                num_groups=num_groups,
                time_features=time_features,
                dropout=dropout,
            )
        ]

        residual_blocks = residual_blocks + [
            ResidualBlockWithTime(
                in_channels=out_channels + in_channels,
                out_channels=out_channels,
                num_groups=num_groups,
                time_features=time_features,
                dropout=dropout,
            )
            for _ in range(num_residual_blocks - 1)
        ]

        self.residual_blocks = torch.nn.ModuleList(residual_blocks)

        self.upsample = torch.nn.Sequential(
            torch.nn.Upsample(scale_factor=2),
            torch.nn.Conv2d(
                in_channels=in_channels,
                out_channels=in_channels,
                kernel_size=5,
                stride=1,
                padding=2,
            ),
        )

    def forward(
        self, x: torch.Tensor, t: torch.Tensor, skips: List[torch.Tensor]
    ) -> torch.Tensor:
        assert len(x.shape) == 4  # B C H W
        assert len(t.shape) == 2  # B F
        assert len(skips) == len(self.residual_blocks)

        x = self.upsample(x)

        for rb, s in zip(self.residual_blocks, reversed(skips)):
            x = torch.concat([x, s], dim=-3)
            x = rb(x, t)

        return x


@torch.no_grad()
def test_upsample_block():
    x = torch.randn(2, 4, 8, 18)
    t = torch.randn(2, 7)

    db = DownsampleBlock(
        in_channels=4,
        out_channels=12,
        time_features=7,
        num_groups=2,
        num_residual_blocks=5,
        dropout=0.1,
    )

    y, skips = db(x, t)

    ub = UpsampleBlock(
        in_channels=12,
        out_channels=4,
        time_features=7,
        num_groups=1,
        num_residual_blocks=5,
        dropout=0.1,
    )

    y = ub(y, t, skips)

    assert y.shape == (2, 4, 8, 18)


test_upsample_block()

### U-Net with time encoding
Finish the implementation of U-Net with time encoding.  
It should consist of downsampling and upsampling paths along with skip connections between them.  
Remember to encode time positionally in the beginning.

In [ ]:
class TimeEncoder(torch.nn.Module):
    def __init__(self, time_features: int, max_time_val: int) -> None:
        super().__init__()

        assert time_features % 2 == 0

        angles = (
            1.0
            / (
                10_000.0
                ** (torch.arange(0, time_features, 2).to(torch.float32) / time_features)
            )[None, :]
        )
        time_steps = torch.arange(0, max_time_val + 1).to(torch.float32)[:, None]

        encodings = angles * time_steps
        encodings = torch.concat([torch.sin(encodings), torch.cos(encodings)], dim=-1)
        assert encodings.shape == (max_time_val + 1, time_features)

        self.register_buffer("encodings", encodings)

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        assert len(t.shape) == 1  # B

        res = self.encodings[t]

        assert res.shape[0] == t.shape[0]
        assert len(res.shape) == 2
        return res


class UNetWithTime(torch.nn.Module):
    def __init__(
        self,
        image_channels: int,
        depth: int,
        max_time_val: int,
        time_features: int,
        num_groups: int = 32,
        dropout: float = 0.05,
        num_residual_blocks=2,
    ):
        super().__init__()
        assert depth > 0

        self.time_encoder = torch.nn.Sequential(
            TimeEncoder(time_features=time_features, max_time_val=max_time_val),
            torch.nn.Linear(time_features, time_features),
            torch.nn.SiLU(),
            torch.nn.Linear(time_features, time_features)
        )
        ch = num_groups * 2
        downsampling = [
            DownsampleBlock(
                in_channels=image_channels,
                out_channels=ch,
                num_groups=num_groups,
                time_features=time_features,
                dropout=dropout,
                num_residual_blocks=num_residual_blocks,
            )
        ]
        for _ in range(depth - 1):
            downsampling.append(
                DownsampleBlock(
                    in_channels=ch,
                    out_channels=2 * ch,
                    num_groups=num_groups,
                    time_features=time_features,
                    dropout=dropout,
                    num_residual_blocks=num_residual_blocks,
                )
            )
            ch = 2 * ch

        self.downsampling = torch.nn.ModuleList(downsampling)

        self.middle = torch.nn.ModuleList(
            [
                ResidualBlockWithTime(
                    in_channels=ch,
                    out_channels=ch,
                    num_groups=num_groups,
                    time_features=time_features,
                    dropout=dropout,
                )
                for _ in range(num_residual_blocks)
            ]
        )

        upsampling = [
            UpsampleBlock(
                in_channels=ch,
                out_channels=ch // 2,
                num_groups=num_groups,
                time_features=time_features,
                dropout=dropout,
                num_residual_blocks=num_residual_blocks,
            )
        ]

        ch = ch // 2
        for _ in range(depth - 1):
            upsampling.append(
                UpsampleBlock(
                    in_channels=ch,
                    out_channels=ch // 2,
                    num_groups=num_groups,
                    time_features=time_features,
                    dropout=dropout,
                    num_residual_blocks=num_residual_blocks,
                )
            )
            ch = ch // 2

        self.upsampling = torch.nn.ModuleList(upsampling)

        self.final = torch.nn.Sequential(
            torch.nn.GroupNorm(num_groups=1, num_channels=ch),
            torch.nn.SiLU(),
            torch.nn.Conv2d(
                in_channels=ch,
                out_channels=image_channels,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        assert len(x.shape) == 4  # B C H W
        assert len(t.shape) == 1  # B

        ### TODO {

        ### }

        return x


@torch.no_grad()
def test_unet_time_block():
    x = torch.randn(2, 4, 32, 32)
    t = torch.tensor([1, 2], dtype=torch.int32)

    ut = UNetWithTime(image_channels=4, time_features=10, depth=3, max_time_val=11)
    y = ut(x, t)

    assert y.shape == x.shape


test_unet_time_block()

### Noise Handling
Finish the implementation of noise handling class.

In [ ]:
class NoiseHandling(torch.nn.Module):
    def __init__(self, max_time: int, beta_start=1e-4, beta_end=0.02) -> None:
        super().__init__()

        self.max_time = max_time

        # for step zero we set beta to 0
        self.register_buffer(
            "betas",
            torch.concat(
                [
                    torch.zeros((1,), dtype=torch.float32),
                    torch.linspace(beta_start, beta_end, max_time),
                ]
            ),
        )
        alphas = 1 - self.betas
        alphas = torch.cumprod(alphas, dim=0)
        self.register_buffer("alphas", alphas)

    @torch.no_grad()
    def add_noise_at_time_t(
        self, x_0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor
    ) -> torch.Tensor:
        """
        Returns x_t sampled from q(x_t | x_0)
        Uses noise as a source of N(0, 1) noise
        """
        assert len(t.shape) == 1 # B
        assert len(x_0.shape) == 4  # B, C, H, W

        ### TODO {


        ### }

        assert sample.shape == x_0.shape
        return sample

    @torch.no_grad()
    def step_back_in_time(
        self, pred_noise: torch.Tensor, x_t: torch.Tensor, t: torch.Tensor
    ) -> torch.Tensor:
        assert len(t.shape) == 1
        assert len(x_t.shape) == 4  # B, C, H, W
        assert x_t.shape == pred_noise.shape
        alpha_t = self.alphas[t][:, None, None, None]
        alpha_tm1 = self.alphas[t - 1][:, None, None, None]
        beta_t = self.betas[t][:, None, None, None]

        """
        Returns x_{t-1} sampled from q(x_{t-1} | x_t, x_0)
        where x_0 is approximated using the noise predicted by the model
        """

        # according to our model
        # x_t = torch.sqrt(alpha_t)*x_0 + torch.sqrt(1-alpha_t)*pred_noise
        # so x_0 = (x_t - torch.sqrt(1-alpha_t)*pred_noise)/torch.sqrt(alpha_t)
        # but we dont want to compute x_0 directly as torch.sqrt(alpha_t) can be very small

        # let us consider q(x_{t-1}|x_t, x_0)
        # (equation 71 in https://arxiv.org/pdf/2208.11970 - note that our alpha_t is their alpha_t_hat
        # and that their alpha_t is our 1-beta_t)

        # plug in x_0, simplify, and finish the procedure

        ### TODO {


        ### }

        return mean + torch.randn_like(pred_noise) * std


@torch.no_grad()
def test_noise_chain():
    x = torch.randn(3, 4, 32, 32)
    t = torch.tensor([0, 1, 2], dtype=torch.int32)

    nc = NoiseHandling(beta_start=0.8, beta_end=0.9, max_time=10)
    y = nc.add_noise_at_time_t(x, t, noise=torch.rand_like(x))

    assert y.shape == x.shape


test_noise_chain()

In [ ]:
@torch.no_grad()
def view_example(input_shape: Tuple[int, int, int], model: UNetWithTime, noise_chain: NoiseHandling, device, epoch:int):
    model.eval()
    batch = 16
    x = torch.randn(size=(batch,)+input_shape, device=device)

    for i in reversed(range(1, noise_chain.max_time + 1)):
        t = torch.tensor([i] * batch, dtype=torch.int32, device=device)
        pred_noise = model(x, t)
        assert x.shape == pred_noise.shape
        x = noise_chain.step_back_in_time(pred_noise=pred_noise, x_t=x, t=t)

    #print(x) # may help to debug the formula above
    grid = torchvision.utils.make_grid(x.cpu(), nrow=4)
    fig = plt.imshow(grid.permute(1, 2, 0))
    fig.figure.savefig(f'./epoch_{epoch}_sample.png')
    fig.figure.savefig(f'./latest.png')

## Training
Finish the implementation of the train step.  
Briefly speaking  
1. Zero grad optimizer
2. Sample timestep for each element of the batch
3. Sample `x_t` from `q` using `add_noise_at_time_t` method of `NoiseHandling`
4. Optimize the noise prediction of the model (use either mse or huber loss)

Note that `view_example` is saving images every epoch.  
The parameters below and the architecture may not be optimal (especially for CIFAR-10).

In [ ]:
DEVICE = torch.device("cuda")
MAX_TIME = 512
TIME_FEATURES = 32
UNET_DEPTH = 3
BETA = 0.02


def create_model_optim():
    construct_model = lambda: UNetWithTime(
        image_channels=IMAGE_CHANNELS,
        depth=4,
        max_time_val=MAX_TIME,
        time_features=TIME_FEATURES,
    )
    model = construct_model()
    model.to(DEVICE)

    optimizer = torch.optim.Adam(params=model.parameters(), lr=1e-4)
    return model, optimizer


def create_noise_handling():
    return NoiseHandling(beta_start=1e-4, beta_end=BETA, max_time=MAX_TIME).to(DEVICE)


def train_step(
    model: UNetWithTime,
    optim: torch.optim.Optimizer,
    input_images: torch.Tensor,
    noise_handling: NoiseHandling,
):

    ### TODO {


    ### }

    return loss.detach()


def train_loop(
    model: UNetWithTime,
    optim: torch.optim.Optimizer,
    noise_handling: NoiseHandling,
    train_loader: torch.utils.data.DataLoader,
    epochs=20,
    gamma=0.0,
):
    model.train()
    for e in range(epochs):
        for step, (x, _) in tqdm(enumerate(train_loader)):
            x = x.to(DEVICE)
            loss = train_step(
                model=model, optim=optim, input_images=x, noise_handling=noise_handling
            )

            if step % 200 == 0:
                print(f"\nEPOCH {e} STEP {step}: Loss {loss}")
                view_example(
                    input_shape=x.shape[1:],
                    model=model,
                    noise_chain=noise_handling,
                    device=DEVICE,
                    epoch=e
                )
                model.train()


model, optimizer = create_model_optim()
noise_handling = create_noise_handling()
train_loop(
    model=model,
    optim=optimizer,
    noise_handling=noise_handling,
    train_loader=TRAIN_LOADER,
)

## Epilog
As mentioned above this is a simplified version of diffusion model from .  
Can you name things that are missing/changed from the original [Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239) paper?  
